In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import json

import pandas as pd

from src import config

# 08 - Post-Processing: Deduplicate `persona_dataset.json`

`results/persona_dataset.json` (stage 4's output) turns out to contain duplicate `content` under distinct `post_id`s -- traced back to `data/raw/posts.parquet` itself, which has many rows that are byte-identical in `content` but carry different `id`s (templated/boilerplate source posts, not a bug in this pipeline's merge step). `exposure.sample_feed()` dedups by row/`post_id`, not by `content`, so the same post text can currently appear more than once in a single sampled feed.

This notebook is read-only against `results/persona_dataset.json` -- it does NOT overwrite it. It writes a separate deduplicated copy, `results/persona_dataset_deduped.json`, so both the original and the deduped view are available side by side.

## Load `results/persona_dataset.json`

In [2]:
path = config.RESULTS_DIR / "persona_dataset.json"
with open(path, "r", encoding="utf-8") as f:
    data = json.load(f)

df = pd.DataFrame(data).rename(columns={"role": "persona"})
print(f"loaded {len(df)} rows from {path}")

loaded 2739 rows from C:\Users\THINKPAD T14\OneDrive\Documents\000 THINKPAD\Projects\Collab\Moltbook-drift-data\moltbook-persona\results\persona_dataset.json


## Dedup on `content`

Drops rows whose `content` exactly matches an earlier row's, keeping the first occurrence (by original file order). Dedup key is `content` alone (not `(persona, category, content)`) -- a handful of duplicate posts span different persona/category pairs too, and those should collapse as well.

In [3]:
before = len(df)
deduped = df.drop_duplicates(subset=["content"], keep="first").reset_index(drop=True)
after = len(deduped)
print(f"before dedup: {before} rows")
print(f"after dedup:  {after} rows")
print(f"removed:      {before - after} duplicate-content rows")

before dedup: 2739 rows
after dedup:  2383 rows
removed:      356 duplicate-content rows


## Persona x category coverage, before vs. after

In [4]:
before_table = df.groupby(["persona", "category"]).size().unstack(fill_value=0)
after_table = deduped.groupby(["persona", "category"]).size().unstack(fill_value=0)

print("before dedup:")
print(before_table)
print("\nafter dedup:")
print(after_table)
print("\ndelta (after - before):")
print(after_table - before_table)

before dedup:
category                  A    B    C    D    E    F
persona                                             
assistant               173  180  178   94  179  147
evil                    101    6    6    4  118   26
malicious-manipulative   56  134   13  183   31  110
other (toxic)            38   49  149   12   46   57
pirate                   50   50   50   50   50   50
poet                     50   50   50   50   50   50
sycophantic               5    3   32    1    4    4

after dedup:
category                  A    B    C    D    E    F
persona                                             
assistant               169  180  166   78  175  147
evil                     79    4    3    3   80   25
malicious-manipulative   51   93    6  165   30  106
other (toxic)            38   47    6   11   45   56
pirate                   50   50   50   50   50   50
poet                     50   50   50   50   50   50
sycophantic               5    3    3    1    4    4

delta (after - be

## Write `results/persona_dataset_deduped.json`

`persona_dataset.json` itself is left untouched -- this is a separate output, not an overwrite.

In [5]:
output_path = config.RESULTS_DIR / "persona_dataset_deduped.json"

# Restore the `role` column name to match persona_dataset.json's original schema.
out_records = deduped.rename(columns={"persona": "role"}).to_dict("records")
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(out_records, f, ensure_ascii=False, indent=2)

print(f"wrote {len(out_records)} rows to {output_path}")

# Sanity check: original file is untouched.
with open(path, "r", encoding="utf-8") as f:
    original_still_intact = json.load(f)
assert len(original_still_intact) == before, "persona_dataset.json changed size -- it should never be written to by this notebook"
print(f"confirmed {path} is unchanged ({len(original_still_intact)} rows)")

wrote 2383 rows to C:\Users\THINKPAD T14\OneDrive\Documents\000 THINKPAD\Projects\Collab\Moltbook-drift-data\moltbook-persona\results\persona_dataset_deduped.json
confirmed C:\Users\THINKPAD T14\OneDrive\Documents\000 THINKPAD\Projects\Collab\Moltbook-drift-data\moltbook-persona\results\persona_dataset.json is unchanged (2739 rows)
